In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from anthropic.types import Message, MessageParam, TextBlock

from learning_journey.claude.client import get_client

client, settings = get_client()

---
## BUILDING THE DATASET
---

In [ ]:
# Starting prompt (version 1)

# An f-string is evaluated the moment Python reaches that line, so it looks up task right then.
# Since task doesn't exist yet, you get a NameError. .format() never gets a chance to run.
PROMPT_TEMPLATE = """
Please provide a solution to the following task:
{task}
"""

# An evaluation dataset contains inputs that we'll feed into our prompt.
# For each combination of prompt and input, we'll run the prompt and analyze the results.
# We can either create this dataset by hand or generate it automatically using Claude.


def add_user_message(messages: list[MessageParam], text: str) -> None:
    user_message: MessageParam = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[MessageParam], text: str) -> None:
    assistant_message: MessageParam = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(
    messages: list[MessageParam],
    *,
    system: str | None = None,
    stop_sequences: list[str] | None = None,
) -> str:
    params = {
        "model": settings.claude_model,
        "max_tokens": settings.default_max_tokens,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response: Message = client.messages.create(**params)
    text = "".join(block.text for block in response.content if isinstance(block, TextBlock))
    return text


# For a production-grade project, the cleaner pattern is a small function. It's explicit,
# testable, and easy to reuse across notebooks and scripts.
def build_prompt(task: str) -> str:
    """Fill the prompt template with a task description."""
    return PROMPT_TEMPLATE.format(task=task)


def generate_dataset() -> object:
    prompt = """
        Generate an evaluation dataset for a prompt evaluation. The dataset will be used
        to evaluate prompts that generate Python, JSON, or Regex specifically for
        AWS-related tasks. Generate an array of JSON objects, each representing task
        that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
          \\{
            "task": "Description of task",
          \\},
          ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function,
        a single JSON object, or a single regex
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
        """

    messages: list[MessageParam] = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [ ]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

In [ ]:
nb_data_dir = Path("data").resolve()
nb_data_dir.mkdir(parents=True, exist_ok=True)
file_path = nb_data_dir / "dataset.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

print(f"Dataset JSON file created at: {file_path}")

---
## RUNNING THE EVAL
---